In [ ]:
from dolfinx import log, default_scalar_type
from dolfinx.fem.petsc import NonlinearProblem
import pyvista
import numpy as np
import ufl
import basix.ufl

from mpi4py import MPI
from dolfinx import fem, mesh, plot


# Geometry

In [ ]:
L = 20.0
domain = mesh.create_box(
    MPI.COMM_WORLD, [[0.0, 0.0, 0.0], [L, 1, 1]], [20, 5, 5], mesh.CellType.hexahedron
)

In [ ]:
def left(x):
    return np.isclose(x[0], 0)


def right(x):
    return np.isclose(x[0], L)


fdim = domain.topology.dim - 1
left_facets = mesh.locate_entities_boundary(domain, fdim, left)
right_facets = mesh.locate_entities_boundary(domain, fdim, right)

In [ ]:
marked_facets = np.hstack([left_facets, right_facets])
marked_values = np.hstack([np.full_like(left_facets, 1), np.full_like(right_facets, 2)])
sorted_facets = np.argsort(marked_facets)
facet_tag = mesh.meshtags(
    domain, fdim, marked_facets[sorted_facets], marked_values[sorted_facets]
)

In [ ]:
u_bc = np.array((0,) * domain.geometry.dim, dtype=default_scalar_type)

# Weak form

In [ ]:
el_u = basix.ufl.element("Lagrange", domain.basix_cell(), 2, shape=(domain.geometry.dim,))
el_p = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
el_mixed = basix.ufl.mixed_element([el_u, el_p])
W = fem.functionspace(domain, el_mixed)

w = fem.Function(W)

In [ ]:
V_u = W.sub(0) # displacement subspace 
V_uCollapsed, V_uCollapsed_to_Vu = V_u.collapse()

u_D = fem.Function(V_uCollapsed)

left_dofs = fem.locate_dofs_topological(V=(V_u, V_uCollapsed), entity_dim=domain.topology.dim - 1, entities=facet_tag.find(1))
bcs = [fem.dirichletbc(u_D, left_dofs, V_u)]

In [ ]:
B = fem.Constant(domain, default_scalar_type((0, 0, 0))) # volume force (gravity, etc)
T = fem.Constant(domain, default_scalar_type((0, 0, -1.5))) # surface force, acting on end

In [ ]:
(u, p) = ufl.split(w)
v_u, v_p = ufl.TestFunctions(W)

In [ ]:
# Spatial dimension
d = len(u)

# Identity tensor
I = ufl.variable(ufl.Identity(d))

# Deformation gradient
F = ufl.variable(I + ufl.grad(u))

# Right Cauchy-Green tensor
C = ufl.variable(F.T * F)

# Invariants of deformation tensors
I_1 = ufl.variable(ufl.tr(C))
J = ufl.variable(ufl.det(F))
I_1_bar = ufl.variable(J**(-2/3) * I_1) 

In [ ]:
E = default_scalar_type(1.0e4)
nu = default_scalar_type(0.3)
mu = fem.Constant(domain, E / (2 * (1 + nu)))
lmbda = fem.Constant(domain, E * nu / ((1 + nu) * (1 - 2 * nu)))

psi = (mu / 2) * (I_1_bar - 3) + p*(J-1) 
P = ufl.diff(psi, F)


In [ ]:
ds = ufl.Measure("ds", domain=domain, subdomain_data=facet_tag, metadata={"quadrature_degree": 4})
dx = ufl.Measure("dx", domain=domain, metadata={"quadrature_degree": 4})

In [ ]:
residual = (
    ufl.inner(ufl.grad(v_u), P) * dx - ufl.inner(v_u, B) * dx - ufl.inner(v_u, T) * ds(2) + ufl.inner((J-1), v_p)*dx
)

# Solving

In [ ]:
petsc_options = {
    "snes_type": "newtonls",
    "snes_linesearch_type": "none",
    "snes_monitor": None,
    "snes_atol": 1e-8,
    "snes_rtol": 1e-8,
    "snes_stol": 1e-8,
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
}
problem = NonlinearProblem(
    residual,
    w,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="hyperelasticity",
)

In [ ]:
log.set_log_level(log.LogLevel.INFO)


problem.solve()

converged = problem.solver.getConvergedReason()
num_its = problem.solver.getIterationNumber()
print(f"Solver convergence: {converged}. Number of iterations {num_its}, Load {T.value}")

# Post-processing, output, plotting

In [ ]:

V_u_out = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1, shape=(3,)))
u_out = fem.Function(V_u_out)
u_out.name = "u"

V_p_out = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1))
p_out = fem.Function(V_p_out)
p_out.name = "p"

u_out.interpolate(w.sub(0).collapse())
p_out.interpolate(w.sub(1).collapse())


In [ ]:
from pathlib import Path
from dolfinx import io
folder = Path("results")
folder.mkdir(exist_ok=True, parents=True)
xdmf = io.XDMFFile(MPI.COMM_WORLD, folder/"mixed_u-p_incompressibleMinimumExample.xdmf", "w")
xdmf.write_mesh(domain)
# xdmf.write_meshtags(facet_tags, domain.geometry)
t=0
xdmf.write_function(p_out, t)
xdmf.write_function(u_out, t)

xdmf.close()